# Code quality - Clearly explained
![](https://miro.medium.com/v2/resize:fit:1400/0*7f4ygDpgLCXHr7U9)


Code quality is an **essential** aspect of software development that can greatly impact the **maintainability**, **reliability**, and **scalability** of your codebase. Python is a popular programming language that **emphasizes readability and ease of use**, making it a great choice for building robust and **maintainable applications**. However, it's essential to follow best practices and principles to ensure that your Python code is of **high quality**.

During this tutorial, we will be exploring some of the most important principles of code quality in Python. We will start by discussing the importance of **writing clean, readable, and maintainable code**. Throughout the tutorial, we will be performing several code refactorings to demonstrate the importance of following these principles. We will take a piece of **poorly written code and progressively transform it into a more elegant, maintainable, and efficient implementation**, using techniques such as refactoring, code reviews, and automated testing. By the end of this tutorial, you will have a solid understanding of how to write high-quality Python code that is easy to read, understand, and maintain, and you will be equipped with the tools and techniques necessary to do so in your own projects.

Let's go???

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from abc import ABC, abstractmethod
import csv
import statistics
import unittest
import tempfile
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/spaceship-titanic/sample_submission.csv
/kaggle/input/spaceship-titanic/train.csv
/kaggle/input/spaceship-titanic/test.csv


# 1. Writing our first class

At this stage, we will be creating a **functional class**, however, it contains various issues related to code quality.

In [2]:
class DataAnalyzer:
    def __init__(self, datafile):
        self.data = pd.read_csv(datafile)

    def get_mean(self, col):
        return np.mean(self.data[col])

    def get_median(self, col):
        return np.median(self.data[col])

    def missing_values(self, col):
        return self.data[col].isnull().sum()

    def remove_outliers(self, col):
        Q1 = np.percentile(self.data[col], 25)
        Q3 = np.percentile(self.data[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        self.data = self.data[(self.data[col] >= lower_bound) & (self.data[col] <= upper_bound)]

So, would you be able to tell me what are the **main problems** of this class?

Here are some issues with this class:

1. **Lack of documentation:** The class and its methods should have docstrings explaining their purpose, input parameters, and return values to help other developers understand and use the class effectively.

1. **Error handling:** The methods do not check for invalid inputs, such as nonexistent column names or incorrect data types. Proper error handling would make the class more robust and user-friendly.

1. **Hardcoded values:** The remove_outliers method uses a hardcoded value (1.5) to calculate the bounds for outliers. This value could be a parameter, allowing users to control the degree of outlier removal.

1. **Modularity:** The class is tightly coupled with pandas and numpy, which might limit its usefulness in projects using different libraries. Creating an interface to work with different data structures and libraries could improve its versatility.

1. **Testing:** There are no test cases to ensure the correctness of the class's functionality. Writing tests helps catch bugs and maintain the codebase.

1. **Flexibility:** The class only supports CSV files as input. Adding support for other file formats or data sources, such as JSON, Excel, or databases, would make the class more versatile.

1. **Single Responsibility Principle (SRP):** The class is responsible for reading data from a file, calculating statistics, and removing outliers. This could be separated **into different classes or modules**, where one class handles file I/O, another class deals with statistical analysis, and another class processes the data.

and so on...

# 2. Lack of documentation

To solve the **"lack of documentation"** problem, you can add docstrings to your class, its methods, and any relevant functions or modules. Docstrings are multi-line comments that provide a clear explanation of the purpose, inputs, and outputs of your code. They make it easier for other developers to understand and use your code effectively. Here's how you can add docstrings to the `DataAnalyzer` class:

In [3]:
class DataAnalyzer:
    """
    A class used to analyze data from a CSV file.
    """

    def __init__(self, datafile):
        """
        Initialize the DataAnalyzer with data from a CSV file.

        Parameters
        ----------
        datafile : str
            The path to the CSV file containing the data.
        """
        self.data = pd.read_csv(datafile)

    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.
        """
        return np.mean(self.data[col])

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.
        """
        return np.median(self.data[col])

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.
        """
        return self.data[col].isnull().sum()

    def remove_outliers(self, col):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.

        Returns
        -------
        None
        """
        Q1 = np.percentile(self.data[col], 25)
        Q3 = np.percentile(self.data[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        self.data = self.data[(self.data[col] >= lower_bound) & (self.data[col] <= upper_bound)]

Now you can see that the class is **clearly documented** (and better understood). Let's move on to the next step...

# 3. Error handling

To solve the "**Error handling**" problem, you can add checks for invalid inputs and raise appropriate exceptions with meaningful error messages. This will make the class more robust and user-friendly. Here's an example of how you can add error handling to the `DataAnalyzer` class:

In [4]:
class DataAnalyzer:
    """
    A class used to analyze data from a CSV file.
    """

    def __init__(self, datafile):
        """
        Initialize the DataAnalyzer with data from a CSV file.

        Parameters
        ----------
        datafile : str
            The path to the CSV file containing the data.
        """
        if not isinstance(datafile, str):
            raise ValueError("datafile should be a string representing the path to the CSV file")

        try:
            self.data = pd.read_csv(datafile)
        except FileNotFoundError:
            raise FileNotFoundError(f"Could not find the CSV file at '{datafile}'")
            
    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.
        """
        return np.mean(self.data[col])

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.
        """
        self._validate_column_name(col)
        return np.median(self.data[col])

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.
        """
        self._validate_column_name(col)
        return self.data[col].isnull().sum()

    def remove_outliers(self, col):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.

        Returns
        -------
        None
        """
        self._validate_column_name(col)
        
        Q1 = np.percentile(self.data[col], 25)
        Q3 = np.percentile(self.data[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        self.data = self.data[(self.data[col] >= lower_bound) & (self.data[col] <= upper_bound)]
        
    def _validate_column_name(self, col):
        """
        Check if the column name exists in the dataset and raise an exception if not.

        Parameters
        ----------
        col : str
            The column name to validate.

        Raises
        ------
        ValueError
            If the column name is not found in the dataset.
        """
        if col not in self.data.columns:
            raise ValueError(f"Column '{col}' not found in the dataset")        

In this example, I added error handling in the `__init__` method to check if the datafile parameter is a string and if the file exists. I also added a private method `_validate_column_name` to check if the given column name exists in the dataset. This method is called at the beginning of each method that accepts a column name as a parameter.

By adding these error checks and raising meaningful exceptions, you make the class more robust and provide clear guidance to users when they encounter issues. Let's move on to the next step...

# 4. Hardcoded values

o handle the "**hardcoded values**" problem, you can replace hardcoded values with parameters that have default values. This allows users to customize these values when needed while maintaining the original behavior by default. Here's an example of how you can modify the remove_outliers method to allow customizing the IQR multiplier:

In [5]:
class DataAnalyzer:
    """
    A class used to analyze data from a CSV file.
    """

    def __init__(self, datafile):
        """
        Initialize the DataAnalyzer with data from a CSV file.

        Parameters
        ----------
        datafile : str
            The path to the CSV file containing the data.
        """
        if not isinstance(datafile, str):
            raise ValueError("datafile should be a string representing the path to the CSV file")

        try:
            self.data = pd.read_csv(datafile)
        except FileNotFoundError:
            raise FileNotFoundError(f"Could not find the CSV file at '{datafile}'")
            
    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.
        """
        return np.mean(self.data[col])

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.
        """
        self._validate_column_name(col)
        return np.median(self.data[col])

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.
        """
        self._validate_column_name(col)
        return self.data[col].isnull().sum()

    def remove_outliers(self, col, iqr_multiplier=1.5):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.
        iqr_multiplier : float, optional, default: 1.5
            The multiplier for the IQR to determine the outlier bounds.

        Returns
        -------
        None
        """
        self._validate_column_name(col)

        if not isinstance(iqr_multiplier, (int, float)):
            raise ValueError("iqr_multiplier should be a numeric value")

        Q1 = np.percentile(self.data[col], 25)
        Q3 = np.percentile(self.data[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - iqr_multiplier * IQR
        upper_bound = Q3 + iqr_multiplier * IQR
        self.data = self.data[(self.data[col] >= lower_bound) & (self.data[col] <= upper_bound)]        
        
    def _validate_column_name(self, col):
        """
        Check if the column name exists in the dataset and raise an exception if not.

        Parameters
        ----------
        col : str
            The column name to validate.

        Raises
        ------
        ValueError
            If the column name is not found in the dataset.
        """
        if col not in self.data.columns:
            raise ValueError(f"Column '{col}' not found in the dataset")    

In this example, I added a new parameter `iqr_multiplier` with a default value of **1.5** to the remove_outliers method. This allows users to change the multiplier if needed. I also added a check to ensure that `iqr_multiplier` is a numeric value.

By adding parameters with default values, you make the class more flexible and customizable without affecting the default behavior. Now let's solve the next problem

# 5. Modularity

To deal with the **"modularity"** problem, you can refactor the class to work with different data structures and libraries by creating an interface or an abstract base class. This allows users to implement their own classes that work with the **same interface**, making the code more versatile and easier to maintain. Here's an example of how you can create an abstract base class for the DataAnalyzer and provide a **pandas-based** and **CSV-based** implementation:

In [6]:
class AbstractDataAnalyzer(ABC):
    """
    An abstract base class for a data analyzer.
    
    This class defines the interface for a data analyzer with abstract methods,
    allowing different implementations using various data structures or libraries.
    """

    @abstractmethod
    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.
        """
        pass

    @abstractmethod
    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.
        """
        pass

    @abstractmethod
    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.
        """
        pass

    @abstractmethod
    def remove_outliers(self, col, iqr_multiplier=1.5):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.
        iqr_multiplier : float, optional, default: 1.5
            The multiplier for the IQR to determine the outlier bounds.

        Returns
        -------
        None
        """
        pass

In [7]:
class PandasDataAnalyzer(AbstractDataAnalyzer):
    """
    A pandas-based implementation of the AbstractDataAnalyzer.
    
    This class provides a concrete implementation of the AbstractDataAnalyzer
    interface using pandas and numpy for data manipulation and statistical analysis.
    """

    def __init__(self, datafile):
        """
        Initialize the DataAnalyzer with data from a CSV file.

        Parameters
        ----------
        datafile : str
            The path to the CSV file containing the data.
        """
        if not isinstance(datafile, str):
            raise ValueError("datafile should be a string representing the path to the CSV file")

        try:
            self.data = pd.read_csv(datafile)
        except FileNotFoundError:
            raise FileNotFoundError(f"Could not find the CSV file at '{datafile}'")
            
    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.
        """
        return np.mean(self.data[col])

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.
        """
        self._validate_column_name(col)
        return np.median(self.data[col])

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.
        """
        self._validate_column_name(col)
        return self.data[col].isnull().sum()

    def remove_outliers(self, col, iqr_multiplier=1.5):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.
        iqr_multiplier : float, optional, default: 1.5
            The multiplier for the IQR to determine the outlier bounds.

        Returns
        -------
        None
        """
        self._validate_column_name(col)

        if not isinstance(iqr_multiplier, (int, float)):
            raise ValueError("iqr_multiplier should be a numeric value")

        Q1 = np.percentile(self.data[col], 25)
        Q3 = np.percentile(self.data[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - iqr_multiplier * IQR
        upper_bound = Q3 + iqr_multiplier * IQR
        self.data = self.data[(self.data[col] >= lower_bound) & (self.data[col] <= upper_bound)]        
        
    def _validate_column_name(self, col):
        """
        Check if the column name exists in the dataset and raise an exception if not.

        Parameters
        ----------
        col : str
            The column name to validate.

        Raises
        ------
        ValueError
            If the column name is not found in the dataset.
        """
        if col not in self.data.columns:
            raise ValueError(f"Column '{col}' not found in the dataset")   

In [8]:
class CsvDataAnalyzer(AbstractDataAnalyzer):
    """
    A CSV-based implementation of the AbstractDataAnalyzer.

    This class provides a concrete implementation of the AbstractDataAnalyzer
    interface using Python's built-in csv and statistics modules for data manipulation
    and statistical analysis.
    """

    def __init__(self, datafile):
        """
        Initialize the CsvDataAnalyzer with data from a CSV file.

        Parameters
        ----------
        datafile : str
            The path to the CSV file containing the data.

        Raises
        ------
        ValueError
            If datafile is not a string.
        FileNotFoundError
            If the file at the specified path is not found.
        """
        if not isinstance(datafile, str):
            raise ValueError("datafile should be a string representing the path to the CSV file")

        self.data = []

        try:
            with open(datafile, newline='') as csvfile:
                reader = csv.DictReader(csvfile)
                for row in reader:
                    self.data.append(row)
        except FileNotFoundError:
            raise FileNotFoundError(f"Could not find the CSV file at '{datafile}'")

    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        numeric_data = [float(row[col]) for row in self.data if row[col] != '']
        return statistics.mean(numeric_data)

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        numeric_data = [float(row[col]) for row in self.data if row[col] != '']
        return statistics.median(numeric_data)

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        return sum(1 for row in self.data if row[col] == '')
    
    def remove_outliers(self, col, iqr_multiplier=1.5):
        """
        Remove the outliers from the specified column using the IQR method.

        Parameters
        ----------
        col : str
            The column name from which to remove the outliers.
        iqr_multiplier : float, optional, default: 1.5
            The multiplier for the IQR to determine the outlier bounds.

        Raises
        ------
        ValueError
            If iqr_multiplier is not a numeric value.
        """
        self._validate_column_name(col)

        if not isinstance(iqr_multiplier, (int, float)):
            raise ValueError("iqr_multiplier should be a numeric value")

        numeric_data = [float(row[col]) for row in self.data if row[col] != '']
        Q1 = statistics.quantiles(numeric_data, n=4)[0]
        Q3 = statistics.quantiles(numeric_data, n=4)[2]
        IQR = Q3 - Q1
        lower_bound = Q1 - iqr_multiplier * IQR
        upper_bound = Q3 + iqr_multiplier * IQR
        self.data = [row for row in self.data if float(row[col]) >= lower_bound and float(row[col]) <= upper_bound]

    def _validate_column_name(self, col):
        """
        Check if the column name exists in the dataset and raise an exception if not.

        Parameters
        ----------
        col : str
            The column name to validate.

        Raises
        ------
        ValueError
            If the column name is not found in the dataset.
        """
        if not self.data or col not in self.data[0]:
            raise ValueError(f"Column '{col}' not found in the dataset")

By creating another subclass of `AbstractDataAnalyzer`, you demonstrate the **modularity** and **flexibility** provided by the abstract base class.

Shall we solve another problem now?

# 6. Testing


To deal with the 'Testing' problem, you can write unit tests for your code using a testing framework like unittest or pytest. Unit tests help ensure that individual parts of your code are working as expected and make it easier to maintain and improve your code.

Here's an example of how you can create unit tests for the CsvDataAnalyzer class using the unittest module:

In [9]:
class TestCsvDataAnalyzer(unittest.TestCase):

    def setUp(self):
        # Create a temporary CSV file for testing
        self.test_file = tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".csv")
        self.test_file.write("A,B,C\n1,2,3\n4,5,6\n7,8,\n")
        self.test_file.close()

        self.analyzer = CsvDataAnalyzer(self.test_file.name)

    def tearDown(self):
        # Remove the temporary CSV file after testing
        os.unlink(self.test_file.name)

    def test_get_mean(self):
        mean = self.analyzer.get_mean("A")
        self.assertAlmostEqual(mean, 4, delta=0.01)

    def test_get_median(self):
        median = self.analyzer.get_median("B")
        self.assertEqual(median, 5)

    def test_missing_values(self):
        missing_values = self.analyzer.missing_values("C")
        self.assertEqual(missing_values, 1)
        

unittest.main(argv=[''], verbosity=2, exit=False)

test_get_mean (__main__.TestCsvDataAnalyzer) ... ok
test_get_median (__main__.TestCsvDataAnalyzer) ... ok
test_missing_values (__main__.TestCsvDataAnalyzer) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.009s

OK


This will execute the test class and display the results within the Notebook. By following these steps, you can run tests for the `CsvDataAnalyzer` class and ensure that each method is working as expected.

Now let's solve one more problem...

# 7. Flexibility

To address the 'Flexibility' problem, you can design your code to be easily extensible, adaptable, and modular. This allows you to accommodate future changes in requirements and easily integrate new features or improvements. 

For example, suppose you now have to deal with files in **json** format, then a solution option would be:

In [10]:
class JsonDataAnalyzer(AbstractDataAnalyzer):
    def __init__(self, datafile):
        """
        Initialize the JsonDataAnalyzer with data from a JSON file.

        Parameters
        ----------
        datafile : str
            The path to the JSON file containing the data.

        Raises
        ------
        FileNotFoundError
            If the specified JSON file does not exist.
        """
        if not os.path.exists(datafile):
            raise FileNotFoundError(f"The specified JSON file does not exist: {datafile}")

        with open(datafile, 'r') as jsonfile:
            self.data = json.load(jsonfile)

    def get_mean(self, col):
        """
        Calculate the mean of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the mean.

        Returns
        -------
        float
            The mean of the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        numeric_data = [float(row[col]) for row in self.data if row[col] != '']
        return statistics.mean(numeric_data)

    def get_median(self, col):
        """
        Calculate the median of the specified column.

        Parameters
        ----------
        col : str
            The column name for which to calculate the median.

        Returns
        -------
        float
            The median of the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        numeric_data = [float(row[col]) for row in self.data if row[col] != '']
        return statistics.median(numeric_data)

    def missing_values(self, col):
        """
        Count the number of missing values in the specified column.

        Parameters
        ----------
        col : str
            The column name for which to count the missing values.

        Returns
        -------
        int
            The number of missing values in the specified column.

        Raises
        ------
        ValueError
            If the specified column name is not found in the dataset.
        """
        self._validate_column_name(col)
        return sum(1 for row in self.data if row[col] == '')

## 7.1 Factory Method - design pattern

The Factory Method pattern provides an interface for creating objects in a super class, but allows subclasses to alter the type of objects that will be created. In this case, we'll create a factory function that takes the input data file and its format as arguments and returns an instance of the appropriate data analyzer class: `CsvDataAnalyzer`, `PandasDataAnalyzer`, or `JsonDataAnalyzer`.

Create a factory function that takes the input data file and its format as arguments and returns an instance of the appropriate data analyzer class:

In [11]:
def create_data_analyzer(datafile, format):
    """
    Factory function to create an instance of the appropriate data analyzer class based on the input data format.

    Parameters
    ----------
    datafile : str
        The path to the data file containing the data.
    format : str
        The format of the input data file (e.g., 'csv', 'json', 'pandas').

    Returns
    -------
    AbstractDataAnalyzer
        An instance of the appropriate data analyzer class.

    Raises
    ------
    ValueError
        If the specified data format is not supported.
    """
    if format.lower() == 'csv':
        return CsvDataAnalyzer(datafile)
    elif format.lower() == 'json':
        return JsonDataAnalyzer(datafile)
    elif format.lower() == 'pandas':
        return PandasDataAnalyzer(datafile)
    else:
        raise ValueError(f"Unsupported data format: {format}")

This approach allows you to create instances of the appropriate data analyzer class based on the input data format without having to explicitly call each class's constructor. It also makes it easy to add support for new data formats or sources by creating new data analyzer classes and updating the `create_data_analyzer` function to instantiate the appropriate class.

By using the Factory Method pattern, you can create a more flexible, modular, and maintainable codebase.

# 8. All together!

Now let's apply what we've learned here to the numeric columns in the training database.

In [12]:
datafile = '/kaggle/input/spaceship-titanic/train.csv'

# Create the data analyzer instance using the factory function
data_analyzer = create_data_analyzer(datafile, format="pandas")
print(type(data_analyzer))

<class '__main__.PandasDataAnalyzer'>


Note that an object of the `PandasDataAnalyzer` class has been instantiated here. 


Now let's show the properties of each of the numerical columns:

In [13]:
for c in ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']:
    print(f"Simple stats for the column: {c}")
    print(f"- Mean: {data_analyzer.get_mean(c)}")
    print(f"- Missing: {data_analyzer.missing_values(c)}")
    print("")

Simple stats for the column: Age
- Mean: 28.82793046746535
- Missing: 179

Simple stats for the column: RoomService
- Mean: 224.687617481203
- Missing: 181

Simple stats for the column: FoodCourt
- Mean: 458.07720329024676
- Missing: 183

Simple stats for the column: ShoppingMall
- Mean: 173.72916912197996
- Missing: 208

Simple stats for the column: Spa
- Mean: 311.1387779083431
- Missing: 183

Simple stats for the column: VRDeck
- Mean: 304.8547912992357
- Missing: 188



# 9. Concluding...


This tutorial demonstrates the importance of applying object-oriented code quality techniques in the context of data science. While data science projects often involve complex data manipulation and analysis tasks, the principles of clean and maintainable code should not be overlooked. By following established design principles and best practices, such as the SOLID principles, data scientists can create more modular, flexible, and extensible code that is easier to understand, debug, and maintain. This is particularly crucial in a field like data science, where code is often shared and built upon by multiple team members and needs to be adaptable to evolving requirements and use cases.

Although the examples presented in this tutorial are relatively simple, the concepts and techniques can be easily extended and applied to more complex data science projects. By employing object-oriented design principles, such as the Single Responsibility Principle and the Open/Closed Principle, as well as design patterns like the Factory Method pattern, data scientists can develop more robust and scalable solutions. This tutorial also emphasizes the importance of proper documentation, error handling, and testing, which are essential aspects of high-quality code in any programming domain. By incorporating these practices into their daily work, data scientists can not only improve the quality of their code but also enhance their ability to collaborate with others and contribute to the growth of the data science field.

Here are some additional points of improvement to consider for your data analyzer classes:

1. Extend error handling: Improve error handling by adding more specific error messages and handling different types of errors, such as incorrect file format, data type errors, or missing required columns.

1. Data validation: Add data validation steps in the data loading process to ensure that the input data meets specific requirements, such as checking for required columns, ensuring data types are as expected, or validating data ranges.

1. Add more data analysis functionality: You can extend the data analysis capabilities by implementing additional methods, such as calculating the mode, standard deviation, or variance. You could also include functionality for data visualization, like generating plots or charts.

1. Add support for column filtering and data transformation: Implement methods that allow users to filter columns, apply transformations, or perform other data preprocessing tasks before analysis.

1. Optimize performance: Look for opportunities to optimize the performance of the data analyzer classes, such as using more efficient algorithms, libraries, or parallel processing techniques for large datasets.

1. Logging: Implement logging to track the execution of your code, monitor performance, and identify any issues or errors.

1. Add configuration options: Provide configuration options that allow users to customize the behavior of the data analyzer classes, such as specifying custom delimiter characters for CSV files, selecting specific columns for analysis, or defining data cleaning rules.

1. Use context managers: When dealing with file I/O, consider using context managers (i.e., the with statement) to ensure that files are properly closed after reading or writing.

By addressing these points, you can further improve the robustness, flexibility, and functionality of your data analyzer classes, making them more useful and adaptable to a wider range of use cases.

![](https://upload.wikimedia.org/wikipedia/commons/thumb/e/ea/Thats_all_folks.svg/1200px-Thats_all_folks.svg.png)